# Quick Challenger for `order = 1`

This notebook runs a lightweight challenger screen for `LightGBM` and `CatBoost` against the existing `HistGradientBoosting` incumbent.

Guardrails enforced here:
- only `cls_train_full.csv` and `cls_test.csv` are loaded
- `cls_val.csv` is never loaded in this notebook
- `HistGradientBoosting` remains the incumbent and is not retrained
- `Logistic Regression` is not retrained
- outputs are written only to new `quick_challenger_*` files
- the final holdout stays locked

## Run Strategy

1. Load full train and test only.
2. Remove leakage columns if present.
3. Prepare categorical and numeric features without one-hot expansion.
4. Run a 100k/100k quick screen for `LightGBM` and `CatBoost`.
5. If `LightGBM Quick` is near the incumbent, optionally run a 300k train plus full-test mid-scale pass.
6. Keep `VAL` locked at all times.

Operational note: a hard wall-clock abort for `CatBoost` is not portable inside a Windows notebook kernel. This notebook therefore uses only a lean 100k/100k CatBoost run with early stopping and flags runtimes above 30 minutes as a technical limitation.

In [1]:
%pip install lightgbm catboost

  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 16.2 MB/s  0:00:00
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   - -------------------------------------- 4.5/100.2 MB 23.3 MB/s eta 0:00:05
   --- ------------------------------------ 10.0/100.2 MB 23.9 MB/s eta 0:00:04
   ----- ---------------------------------- 14.7/100.2 MB 23.0 MB/s eta 0:00:04
   ------- -------------------------------- 19.9/100.2 MB 24.0 MB/s eta 0:00:04
   ---------- ----------------------------- 25.4/100.2 MB 24.2 MB/s eta 0:00:04
   ------------ --------------------------- 32.2/100.2 MB 25.2 MB/s eta 0:00:03
   --------------- ------------------------ 37.7/100.2 MB 25.7 MB/s eta 0:00:03
   ----------------- ---------------------- 42.7/100.2 MB 25.4 MB/s eta 0:00:03
   -----


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 30)

print("Imports complete.")

Imports complete.


In [28]:
class CatBoostTimeLimitCallback:
    def __init__(self, max_seconds):
        self.max_seconds = max_seconds
        self.start_time = time.perf_counter()
        self.limit_hit = False

    def after_iteration(self, info):
        elapsed = time.perf_counter() - self.start_time
        if elapsed > self.max_seconds:
            self.limit_hit = True
            return False
        return True

## Configuration

All paths, flags, thresholds, benchmark values, and output files are fixed here.

In [35]:
BASE_DIR = Path(r"c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS")
DATA_DIR = BASE_DIR / "Daten"
TRAIN_PATH = DATA_DIR / "cls_train_full.csv"
TEST_PATH = DATA_DIR / "cls_test.csv"
VAL_PATH = DATA_DIR / "cls_val.csv"
INCUMBENT_PATH = BASE_DIR / "full_scale_cls_best_models.csv"

DEFAULT_METRICS_PATH = BASE_DIR / "quick_challenger_catboost_midscale_metrics_default.csv"
THRESHOLD_RESULTS_PATH = BASE_DIR / "quick_challenger_catboost_midscale_threshold_tuning_results.csv"
BEST_MODELS_PATH = BASE_DIR / "quick_challenger_catboost_midscale_best_model.csv"
PREDICTIONS_PATH = BASE_DIR / "quick_challenger_catboost_midscale_predictions.csv"
FEATURE_IMPORTANCE_PATH = BASE_DIR / "quick_challenger_catboost_midscale_feature_importance.csv"

TARGET_COLUMN = "order"
RANDOM_STATE = 42
THRESHOLDS = np.round(np.arange(5, 81) / 100, 2)
EXPECTED_THRESHOLD_COUNT = 76

RUN_LIGHTGBM_QUICK = False
RUN_LIGHTGBM_MID_SCALE_IF_QUICK_PASS = False
FORCE_LIGHTGBM_MID_SCALE = False
RUN_CATBOOST_QUICK = False
RUN_CATBOOST_MID_SCALE = True
RUN_FINAL_VAL = False

TRAIN_QUICK_SAMPLE_SIZE = 100_000
TEST_QUICK_SAMPLE_SIZE = 100_000
TRAIN_MID_SCALE_SIZE = 300_000
CATBOOST_MAX_MINUTES = 30
CATBOOST_MID_SCALE_MAX_FIT_SECONDS = 15 * 60

LEAKAGE_COLUMNS = [
    "lineID",
    "revenue",
    "quantity",
    "q_raw",
    "quantity_class",
    "qty_suspicious",
    "click",
    "basket",
]

CATEGORICAL_CANDIDATES = [
    "salesIndex",
    "category_norm",
    "pharmForm_norm",
    "has_campaign",
    "pid_segment",
    "group12",
    "group34",
    "price_diff_bin",
    "discount_bin",
]

BEST_THRESHOLD_FLOORS = {
    "precision": 0.34,
    "recall": 0.70,
}

QUICK_INTERESTING_FLOORS = {
    "f1": 0.475,
    "mcc": 0.265,
    "auc": 0.715,
    "precision": 0.34,
    "recall": 0.70,
}

CATBOOST_MID_SCALE_REPLACEMENT_RULE = {
    "f1": 0.482480,
    "mcc": 0.274325,
    "auc": 0.719114,
    "auc_tolerance": 0.002,
    "precision": 0.36,
    "recall": 0.70,
    "max_logloss_delta": 0.01,
    "clear_gain_margin": 0.003,
}

INCUMBENT_EXPECTED = {
    "model": "HistGradientBoosting",
    "threshold": 0.22,
    "auc": 0.719114,
    "logloss": 0.493439,
    "f1": 0.482480,
    "precision": 0.360131,
    "recall": 0.730735,
    "mcc": 0.274325,
    "tn": 156832,
    "fp": 108809,
    "fn": 22566,
    "tp": 61240,
}

assert len(THRESHOLDS) == EXPECTED_THRESHOLD_COUNT

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Incumbent path: {INCUMBENT_PATH}")
print(f"Threshold count: {len(THRESHOLDS)}")
print(f"Mid-scale train size: {TRAIN_MID_SCALE_SIZE:,}")
print(f"CatBoost mid-scale max fit seconds: {CATBOOST_MID_SCALE_MAX_FIT_SECONDS}")
print(f"Flags: RUN_LIGHTGBM_QUICK={RUN_LIGHTGBM_QUICK}, RUN_LIGHTGBM_MID_SCALE_IF_QUICK_PASS={RUN_LIGHTGBM_MID_SCALE_IF_QUICK_PASS}, FORCE_LIGHTGBM_MID_SCALE={FORCE_LIGHTGBM_MID_SCALE}, RUN_CATBOOST_QUICK={RUN_CATBOOST_QUICK}, RUN_CATBOOST_MID_SCALE={RUN_CATBOOST_MID_SCALE}, RUN_FINAL_VAL={RUN_FINAL_VAL}")

Train path: c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS\Daten\cls_train_full.csv
Test path: c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS\Daten\cls_test.csv
Incumbent path: c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS\full_scale_cls_best_models.csv
Threshold count: 76
Mid-scale train size: 300,000
CatBoost mid-scale max fit seconds: 900
Flags: RUN_LIGHTGBM_QUICK=False, RUN_LIGHTGBM_MID_SCALE_IF_QUICK_PASS=False, FORCE_LIGHTGBM_MID_SCALE=False, RUN_CATBOOST_QUICK=False, RUN_CATBOOST_MID_SCALE=True, RUN_FINAL_VAL=False


In [39]:
incumbent_df = pd.read_csv(INCUMBENT_PATH)
incumbent_match = incumbent_df.loc[incumbent_df["model"].eq(INCUMBENT_EXPECTED["model"])].copy()
assert len(incumbent_match) == 1, f"Expected exactly one incumbent row, found {len(incumbent_match)}"
incumbent_row = incumbent_match.iloc[0].copy()

for metric_name, expected_value in INCUMBENT_EXPECTED.items():
    actual_value = incumbent_row[metric_name]
    if isinstance(expected_value, str):
        assert actual_value == expected_value, f"Incumbent mismatch for {metric_name}: {actual_value!r}"
    elif metric_name in {"tn", "fp", "fn", "tp"}:
        assert int(actual_value) == int(expected_value), f"Incumbent mismatch for {metric_name}: {actual_value}"
    else:
        assert np.isclose(float(actual_value), float(expected_value), atol=1e-6, rtol=0.0), (
            f"Incumbent mismatch for {metric_name}: expected {expected_value}, got {actual_value}"
        )

print("Validated incumbent HistGradientBoosting row:")
print(pd.DataFrame([incumbent_row]).to_string(index=False))

Validated incumbent HistGradientBoosting row:
               model  threshold      auc  logloss  accuracy  precision   recall      f1      mcc     tn     fp    fn    tp                backend  fit_seconds  precision_collapse_vs_050  precision_ratio_vs_050                                       precision_check_message  is_recommended_winner
HistGradientBoosting       0.22 0.719114 0.493439  0.624049   0.360131 0.730735 0.48248 0.274325 156832 108809 22566 61240 hist_gradient_boosting    27.134214                      False                0.669742 No precision collapse detected relative to the 0.50 baseline.                   True


## Data Load and Leakage Check

Only `cls_train_full.csv` and `cls_test.csv` are loaded. `cls_val.csv` remains untouched.

In [40]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

assert TARGET_COLUMN in train_df.columns and TARGET_COLUMN in test_df.columns
assert set(pd.Series(train_df[TARGET_COLUMN]).dropna().astype(int).unique()) == {0, 1}
assert set(pd.Series(test_df[TARGET_COLUMN]).dropna().astype(int).unique()) == {0, 1}

train_leakage_present = [column for column in LEAKAGE_COLUMNS if column in train_df.columns]
test_leakage_present = [column for column in LEAKAGE_COLUMNS if column in test_df.columns]
removed_leakage = sorted(set(train_leakage_present) | set(test_leakage_present))
already_absent_leakage = [column for column in LEAKAGE_COLUMNS if column not in removed_leakage]

train_feature_columns = [column for column in train_df.columns if column != TARGET_COLUMN and column not in LEAKAGE_COLUMNS]
test_feature_columns = [column for column in test_df.columns if column != TARGET_COLUMN and column not in LEAKAGE_COLUMNS]
assert train_feature_columns == test_feature_columns, "Train and test features differ after leakage removal."

feature_columns = train_feature_columns.copy()
categorical_features = [column for column in CATEGORICAL_CANDIDATES if column in feature_columns]
numeric_features = [column for column in feature_columns if column not in categorical_features]

model_train_df = train_df[feature_columns + [TARGET_COLUMN]].copy()
model_test_df = test_df[feature_columns + [TARGET_COLUMN]].copy()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Feature count after leakage removal: {len(feature_columns)}")
print(f"Categorical features: {categorical_features}")
print(f"Numeric feature count: {len(numeric_features)}")
print(f"Removed leakage columns: {removed_leakage}")
print(f"Already absent leakage columns: {already_absent_leakage}")
print("\nTarget distribution train:")
print(train_df[TARGET_COLUMN].value_counts(dropna=False).sort_index().to_string())
print("\nTarget distribution test:")
print(test_df[TARGET_COLUMN].value_counts(dropna=False).sort_index().to_string())
print(f"\nTrain order rate: {train_df[TARGET_COLUMN].mean():.4%}")
print(f"Test order rate: {test_df[TARGET_COLUMN].mean():.4%}")

Train shape: (1521260, 29)
Test shape: (349447, 29)
Feature count after leakage removal: 28
Categorical features: ['salesIndex', 'category_norm', 'pharmForm_norm', 'has_campaign', 'pid_segment', 'group12', 'group34', 'price_diff_bin', 'discount_bin']
Numeric feature count: 19
Removed leakage columns: []
Already absent leakage columns: ['lineID', 'revenue', 'quantity', 'q_raw', 'quantity_class', 'qty_suspicious', 'click', 'basket']

Target distribution train:
order
0    1182841
1     338419

Target distribution test:
order
0    265641
1     83806

Train order rate: 22.2460%
Test order rate: 23.9825%


## Helper Functions

The helpers below handle sampling, feature preparation, evaluation, threshold tuning, persistence, and comparison to the incumbent.

In [36]:
DEFAULT_METRICS_COLUMNS = [
    "model",
    "model_family",
    "run_label",
    "eval_scope",
    "threshold",
    "auc",
    "logloss",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "mcc",
    "tn",
    "fp",
    "fn",
    "tp",
    "fit_seconds",
    "train_rows",
    "eval_rows",
]

THRESHOLD_RESULTS_COLUMNS = DEFAULT_METRICS_COLUMNS.copy()

BEST_MODELS_COLUMNS = DEFAULT_METRICS_COLUMNS + [
    "eligible_precision_recall",
    "precision_collapse",
    "selection_note",
    "interesting_vs_histgb",
    "interesting_gate_message",
]

PREDICTIONS_COLUMNS = [
    "model",
    "model_family",
    "run_label",
    "eval_scope",
    "row_index",
    TARGET_COLUMN,
    "p_order_1",
    "pred_default_050",
    "best_threshold",
    "pred_best_threshold",
]

FEATURE_IMPORTANCE_COLUMNS = [
    "model",
    "model_family",
    "run_label",
    "feature",
    "importance",
]

default_metrics_rows = []
threshold_result_frames = []
best_model_rows = []
prediction_frames = []
feature_importance_frames = []
model_status_rows = []

def safe_logloss(y_true, y_prob):
    clipped_prob = np.clip(np.asarray(y_prob, dtype=float), 1e-15, 1 - 1e-15)
    return float(log_loss(y_true, clipped_prob))

def sample_stratified_frame(df, target_column, sample_size, random_state):
    if sample_size is None or sample_size >= len(df):
        return df.copy()
    sampled_df, _ = train_test_split(
        df,
        train_size=sample_size,
        stratify=df[target_column],
        random_state=random_state,
    )
    return sampled_df.sort_index().copy()

def compute_threshold_metrics(
    y_true,
    y_prob,
    threshold,
    model_name,
    model_family,
    run_label,
    eval_scope,
    auc_value,
    logloss_value,
    fit_seconds,
    train_rows,
    eval_rows,
):
    y_pred = (np.asarray(y_prob) >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "model_family": model_family,
        "run_label": run_label,
        "eval_scope": eval_scope,
        "threshold": float(threshold),
        "auc": float(auc_value),
        "logloss": float(logloss_value),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fit_seconds": float(fit_seconds),
        "train_rows": int(train_rows),
        "eval_rows": int(eval_rows),
    }

def evaluate_threshold_grid(
    y_true,
    y_prob,
    model_name,
    model_family,
    run_label,
    eval_scope,
    auc_value,
    logloss_value,
    fit_seconds,
    train_rows,
    eval_rows,
    thresholds,
):
    rows = [
        compute_threshold_metrics(
            y_true=y_true,
            y_prob=y_prob,
            threshold=threshold,
            model_name=model_name,
            model_family=model_family,
            run_label=run_label,
            eval_scope=eval_scope,
            auc_value=auc_value,
            logloss_value=logloss_value,
            fit_seconds=fit_seconds,
            train_rows=train_rows,
            eval_rows=eval_rows,
        )
        for threshold in thresholds
    ]
    return pd.DataFrame(rows, columns=THRESHOLD_RESULTS_COLUMNS)

def rank_by_f1(df):
    return df.sort_values(
        by=["f1", "mcc", "precision", "recall", "threshold", "model"],
        ascending=[False, False, False, False, True, True],
    ).reset_index(drop=True)

def rank_by_mcc(df):
    return df.sort_values(
        by=["mcc", "f1", "precision", "recall", "threshold", "model"],
        ascending=[False, False, False, False, True, True],
    ).reset_index(drop=True)

def select_best_threshold(threshold_df, default_row):
    eligible_mask = (
        threshold_df["precision"] >= BEST_THRESHOLD_FLOORS["precision"]
    ) & (
        threshold_df["recall"] >= BEST_THRESHOLD_FLOORS["recall"]
    )

    if eligible_mask.any():
        candidate_df = threshold_df.loc[eligible_mask].copy()
        selection_note = "Selected from thresholds meeting precision and recall floors."
    else:
        candidate_df = threshold_df.copy()
        selection_note = "No threshold met both precision and recall floors; selected best available threshold."

    best_row = rank_by_f1(candidate_df).iloc[0].to_dict()

    precision_collapse = bool(
        (best_row["precision"] < BEST_THRESHOLD_FLOORS["precision"])
        or (
            ((best_row["f1"] > default_row["f1"]) or (best_row["mcc"] > default_row["mcc"]))
            and (best_row["recall"] > default_row["recall"] + 0.03)
            and (best_row["precision"] < default_row["precision"] - 0.03)
        )
    )

    best_row["eligible_precision_recall"] = bool(
        (best_row["precision"] >= BEST_THRESHOLD_FLOORS["precision"])
        and (best_row["recall"] >= BEST_THRESHOLD_FLOORS["recall"])
    )
    best_row["precision_collapse"] = precision_collapse
    best_row["selection_note"] = selection_note
    return best_row

def quick_gate_against_incumbent(best_row):
    checks = {
        "f1_ge_floor": float(best_row["f1"]) >= QUICK_INTERESTING_FLOORS["f1"],
        "mcc_ge_floor": float(best_row["mcc"]) >= QUICK_INTERESTING_FLOORS["mcc"],
        "auc_ge_floor": float(best_row["auc"]) >= QUICK_INTERESTING_FLOORS["auc"],
        "precision_ge_floor": float(best_row["precision"]) >= QUICK_INTERESTING_FLOORS["precision"],
        "recall_ge_floor": float(best_row["recall"]) >= QUICK_INTERESTING_FLOORS["recall"],
        "no_precision_collapse": not bool(best_row["precision_collapse"]),
    }
    passed = all(checks.values())
    failed_checks = [name for name, passed_flag in checks.items() if not passed_flag]
    message = (
        "Quick challenger is close enough to the incumbent for further testing."
        if passed
        else "Quick challenger is not close enough to the incumbent. Failed checks: " + ", ".join(failed_checks)
    )
    return {
        "interesting_vs_histgb": passed,
        "interesting_gate_message": message,
    }

def prepare_lightgbm_frames(train_frame, eval_frame, feature_cols, categorical_cols, numeric_cols):
    X_train = train_frame[feature_cols].copy()
    X_eval = eval_frame[feature_cols].copy()
    imputation_rows = []

    for column in categorical_cols:
        train_values = X_train[column].astype("string").fillna("MISSING")
        eval_values = X_eval[column].astype("string").fillna("MISSING")
        categories = pd.Index(pd.concat([train_values, eval_values], ignore_index=True).drop_duplicates())
        if "MISSING" not in categories:
            categories = categories.append(pd.Index(["MISSING"]))
        categorical_dtype = pd.CategoricalDtype(categories=categories)
        X_train[column] = pd.Series(train_values, index=X_train.index, dtype=categorical_dtype)
        X_eval[column] = pd.Series(eval_values, index=X_eval.index, dtype=categorical_dtype)

    for column in numeric_cols:
        train_numeric = pd.to_numeric(X_train[column], errors="coerce")
        eval_numeric = pd.to_numeric(X_eval[column], errors="coerce")
        fill_value = float(train_numeric.median()) if train_numeric.notna().any() else 0.0
        train_missing = int(train_numeric.isna().sum())
        eval_missing = int(eval_numeric.isna().sum())
        X_train[column] = train_numeric.fillna(fill_value).astype("float32")
        X_eval[column] = eval_numeric.fillna(fill_value).astype("float32")
        imputation_rows.append({
            "feature": column,
            "fill_value": fill_value,
            "train_missing_before": train_missing,
            "eval_missing_before": eval_missing,
        })

    return X_train, X_eval, pd.DataFrame(imputation_rows)

def prepare_catboost_frames(train_frame, eval_frame, feature_cols, categorical_cols, numeric_cols):
    X_train = train_frame[feature_cols].copy()
    X_eval = eval_frame[feature_cols].copy()
    imputation_rows = []

    for column in categorical_cols:
        X_train[column] = X_train[column].astype("string").fillna("MISSING")
        X_eval[column] = X_eval[column].astype("string").fillna("MISSING")

    for column in numeric_cols:
        train_numeric = pd.to_numeric(X_train[column], errors="coerce")
        eval_numeric = pd.to_numeric(X_eval[column], errors="coerce")
        fill_value = float(train_numeric.median()) if train_numeric.notna().any() else 0.0
        train_missing = int(train_numeric.isna().sum())
        eval_missing = int(eval_numeric.isna().sum())
        X_train[column] = train_numeric.fillna(fill_value).astype("float32")
        X_eval[column] = eval_numeric.fillna(fill_value).astype("float32")
        imputation_rows.append({
            "feature": column,
            "fill_value": fill_value,
            "train_missing_before": train_missing,
            "eval_missing_before": eval_missing,
        })

    return X_train, X_eval, pd.DataFrame(imputation_rows)

def build_feature_importance_frame(model_name, model_family, run_label, estimator, feature_names):
    if hasattr(estimator, "feature_importances_"):
        importance_values = estimator.feature_importances_
    elif hasattr(estimator, "get_feature_importance"):
        importance_values = estimator.get_feature_importance()
    else:
        return pd.DataFrame(columns=FEATURE_IMPORTANCE_COLUMNS)

    importance_df = pd.DataFrame({
        "model": model_name,
        "model_family": model_family,
        "run_label": run_label,
        "feature": feature_names,
        "importance": importance_values,
    })
    return importance_df.sort_values(by="importance", ascending=False).reset_index(drop=True)

def persist_outputs():
    default_metrics_df = pd.DataFrame(default_metrics_rows, columns=DEFAULT_METRICS_COLUMNS)
    threshold_results_df = (
        pd.concat(threshold_result_frames, ignore_index=True, sort=False)
        if threshold_result_frames
        else pd.DataFrame(columns=THRESHOLD_RESULTS_COLUMNS)
    )
    best_models_df = pd.DataFrame(best_model_rows, columns=BEST_MODELS_COLUMNS)
    predictions_df = (
        pd.concat(prediction_frames, ignore_index=True, sort=False)
        if prediction_frames
        else pd.DataFrame(columns=PREDICTIONS_COLUMNS)
    )
    feature_importance_df = (
        pd.concat(feature_importance_frames, ignore_index=True, sort=False)
        if feature_importance_frames
        else pd.DataFrame(columns=FEATURE_IMPORTANCE_COLUMNS)
    )

    DEFAULT_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    default_metrics_df.to_csv(DEFAULT_METRICS_PATH, index=False)
    threshold_results_df.to_csv(THRESHOLD_RESULTS_PATH, index=False)
    best_models_df.to_csv(BEST_MODELS_PATH, index=False)
    predictions_df.to_csv(PREDICTIONS_PATH, index=False)
    feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

    return default_metrics_df, threshold_results_df, best_models_df, predictions_df, feature_importance_df

def evaluate_and_store_model(
    model_name,
    model_family,
    run_label,
    eval_scope,
    estimator,
    X_eval,
    y_eval,
    eval_index,
    fit_seconds,
    train_rows,
    feature_names,
):
    y_prob = estimator.predict_proba(X_eval)[:, 1]
    auc_value = float(roc_auc_score(y_eval, y_prob))
    logloss_value = safe_logloss(y_eval, y_prob)
    eval_rows = len(y_eval)

    default_row = compute_threshold_metrics(
        y_true=y_eval,
        y_prob=y_prob,
        threshold=0.50,
        model_name=model_name,
        model_family=model_family,
        run_label=run_label,
        eval_scope=eval_scope,
        auc_value=auc_value,
        logloss_value=logloss_value,
        fit_seconds=fit_seconds,
        train_rows=train_rows,
        eval_rows=eval_rows,
    )
    default_metrics_rows.append(default_row)

    threshold_df = evaluate_threshold_grid(
        y_true=y_eval,
        y_prob=y_prob,
        model_name=model_name,
        model_family=model_family,
        run_label=run_label,
        eval_scope=eval_scope,
        auc_value=auc_value,
        logloss_value=logloss_value,
        fit_seconds=fit_seconds,
        train_rows=train_rows,
        eval_rows=eval_rows,
        thresholds=THRESHOLDS,
    )
    threshold_result_frames.append(threshold_df)

    best_row = select_best_threshold(threshold_df, default_row)
    best_row.update(quick_gate_against_incumbent(best_row))
    best_model_rows.append(best_row)

    prediction_df = pd.DataFrame({
        "model": model_name,
        "model_family": model_family,
        "run_label": run_label,
        "eval_scope": eval_scope,
        "row_index": np.asarray(eval_index, dtype=int),
        TARGET_COLUMN: np.asarray(y_eval, dtype=int),
        "p_order_1": y_prob,
        "pred_default_050": (y_prob >= 0.50).astype(int),
        "best_threshold": float(best_row["threshold"]),
        "pred_best_threshold": (y_prob >= float(best_row["threshold"])).astype(int),
    })
    prediction_frames.append(prediction_df)

    feature_importance_df = build_feature_importance_frame(
        model_name=model_name,
        model_family=model_family,
        run_label=run_label,
        estimator=estimator,
        feature_names=feature_names,
    )
    if not feature_importance_df.empty:
        feature_importance_frames.append(feature_importance_df)

    persist_outputs()
    return default_row, threshold_df, best_row, feature_importance_df

persist_outputs()
print("Output files initialised.")

Output files initialised.


In [37]:
def catboost_midscale_replacement_decision(best_row, incumbent_row):
    best_f1 = float(best_row["f1"])
    best_mcc = float(best_row["mcc"])
    best_auc = float(best_row["auc"])
    best_logloss = float(best_row["logloss"])
    best_precision = float(best_row["precision"])
    best_recall = float(best_row["recall"])

    incumbent_f1 = float(incumbent_row["f1"])
    incumbent_mcc = float(incumbent_row["mcc"])
    incumbent_auc = float(incumbent_row["auc"])
    incumbent_logloss = float(incumbent_row["logloss"])

    delta_f1 = best_f1 - incumbent_f1
    delta_mcc = best_mcc - incumbent_mcc
    delta_auc = best_auc - incumbent_auc
    delta_logloss = best_logloss - incumbent_logloss

    auc_ok = (
        best_auc >= CATBOOST_MID_SCALE_REPLACEMENT_RULE["auc"]
        or (
            best_auc >= CATBOOST_MID_SCALE_REPLACEMENT_RULE["auc"] - CATBOOST_MID_SCALE_REPLACEMENT_RULE["auc_tolerance"]
            and delta_f1 >= CATBOOST_MID_SCALE_REPLACEMENT_RULE["clear_gain_margin"]
            and delta_mcc >= CATBOOST_MID_SCALE_REPLACEMENT_RULE["clear_gain_margin"]
        )
    )

    checks = {
        "f1_strictly_better": best_f1 > CATBOOST_MID_SCALE_REPLACEMENT_RULE["f1"],
        "mcc_strictly_better": best_mcc > CATBOOST_MID_SCALE_REPLACEMENT_RULE["mcc"],
        "auc_rule": auc_ok,
        "logloss_not_clearly_worse": delta_logloss <= CATBOOST_MID_SCALE_REPLACEMENT_RULE["max_logloss_delta"],
        "precision_floor": best_precision >= CATBOOST_MID_SCALE_REPLACEMENT_RULE["precision"],
        "recall_floor": best_recall >= CATBOOST_MID_SCALE_REPLACEMENT_RULE["recall"],
        "no_precision_collapse": not bool(best_row.get("precision_collapse", False)),
    }

    replace_histgb = all(checks.values())
    failed_checks = [name for name, passed_flag in checks.items() if not passed_flag]
    decision_message = (
        "CatBoost Mid-Scale clears the incumbent replacement rule."
        if replace_histgb
        else "CatBoost Mid-Scale does not clear the incumbent replacement rule. Failed checks: " + ", ".join(failed_checks)
    )

    return {
        "replace_histgb": replace_histgb,
        "critical_logloss": delta_logloss > CATBOOST_MID_SCALE_REPLACEMENT_RULE["max_logloss_delta"],
        "delta_f1": delta_f1,
        "delta_mcc": delta_mcc,
        "delta_auc": delta_auc,
        "delta_logloss": delta_logloss,
        "decision_message": decision_message,
    }

## Quick Challenger Runs

`LightGBM` runs first. A mid-scale `LightGBM` pass is attempted only if the quick screen is close enough to the incumbent. `CatBoost` remains sample-only in this notebook.

In [41]:
train_mid_df = sample_stratified_frame(model_train_df, TARGET_COLUMN, TRAIN_MID_SCALE_SIZE, RANDOM_STATE)

print(f"Mid-scale train sample rows: {len(train_mid_df):,}")
print(f"Full test rows: {len(model_test_df):,}")

if RUN_LIGHTGBM_QUICK or RUN_LIGHTGBM_MID_SCALE_IF_QUICK_PASS or FORCE_LIGHTGBM_MID_SCALE:
    raise RuntimeError("All LightGBM paths must remain disabled for the requested CatBoost mid-scale run.")
else:
    model_status_rows.append({
        "model": "LightGBM Quick",
        "run_label": "lightgbm_quick",
        "status": "skipped",
        "fit_seconds": np.nan,
        "note": "LightGBM retraining is disabled for this CatBoost mid-scale run.",
    })
    model_status_rows.append({
        "model": "LightGBM Mid-Scale",
        "run_label": "lightgbm_mid_scale",
        "status": "skipped",
        "fit_seconds": np.nan,
        "note": "LightGBM mid-scale is disabled for this CatBoost mid-scale run.",
    })

if RUN_CATBOOST_QUICK:
    raise RuntimeError("RUN_CATBOOST_QUICK must be False for the requested CatBoost mid-scale run.")
else:
    model_status_rows.append({
        "model": "CatBoost Quick",
        "run_label": "catboost_quick",
        "status": "skipped",
        "fit_seconds": np.nan,
        "note": "CatBoost quick is disabled for this CatBoost mid-scale run.",
    })

if RUN_CATBOOST_MID_SCALE:
    catboost_mid_status = {
        "model": "CatBoost Mid-Scale",
        "run_label": "catboost_mid_scale",
        "status": "not_started",
        "fit_seconds": np.nan,
        "note": "",
    }
    try:
        X_train_mid_cb, X_test_full_cb, catboost_mid_imputation_df = prepare_catboost_frames(
            train_mid_df,
            model_test_df,
            feature_columns,
            categorical_features,
            numeric_features,
        )
        y_train_mid = train_mid_df[TARGET_COLUMN].astype(int).to_numpy()
        y_test_full = model_test_df[TARGET_COLUMN].astype(int).to_numpy()

        catboost_timeout_callback = CatBoostTimeLimitCallback(CATBOOST_MID_SCALE_MAX_FIT_SECONDS)
        catboost_mid_estimator = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=200,
            learning_rate=0.05,
            depth=6,
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            early_stopping_rounds=30,
            thread_count=-1,
            verbose=50,
            allow_writing_files=False,
        )

        fit_start = time.perf_counter()
        catboost_mid_estimator.fit(
            X_train_mid_cb,
            y_train_mid,
            cat_features=categorical_features,
            eval_set=(X_test_full_cb, y_test_full),
            use_best_model=True,
            callbacks=[catboost_timeout_callback],
        )
        catboost_mid_fit_seconds = time.perf_counter() - fit_start

        if catboost_timeout_callback.limit_hit:
            raise TimeoutError(
                f"CatBoost mid-scale fit exceeded {CATBOOST_MID_SCALE_MAX_FIT_SECONDS} seconds and was aborted as a technical limitation."
            )

        _, _, catboost_mid_best_row, _ = evaluate_and_store_model(
            model_name="CatBoost Mid-Scale",
            model_family="CatBoost",
            run_label="catboost_mid_scale",
            eval_scope="full_test",
            estimator=catboost_mid_estimator,
            X_eval=X_test_full_cb,
            y_eval=y_test_full,
            eval_index=model_test_df.index.to_numpy(),
            fit_seconds=catboost_mid_fit_seconds,
            train_rows=len(train_mid_df),
            feature_names=feature_columns,
        )

        catboost_mid_decision = catboost_midscale_replacement_decision(catboost_mid_best_row, incumbent_row)
        catboost_mid_status.update({
            "status": "success",
            "fit_seconds": catboost_mid_fit_seconds,
            "note": catboost_mid_decision["decision_message"],
        })
        print("CatBoost Mid-Scale numeric imputations (top 10):")
        print(catboost_mid_imputation_df.head(10).to_string(index=False))
        print("\nCatBoost Mid-Scale best threshold row:")
        print(pd.DataFrame([catboost_mid_best_row]).to_string(index=False))
        print("\nCatBoost Mid-Scale replacement decision:")
        print(pd.DataFrame([catboost_mid_decision]).to_string(index=False))
    except TimeoutError as exc:
        catboost_mid_status.update({
            "status": "failed_time_limit",
            "note": str(exc),
        })
        print(f"CatBoost Mid-Scale aborted due to time limit: {exc}")
    except Exception as exc:
        catboost_mid_status.update({
            "status": "failed",
            "note": str(exc),
        })
        print(f"CatBoost Mid-Scale failed: {exc}")
    model_status_rows.append(catboost_mid_status)
else:
    model_status_rows.append({
        "model": "CatBoost Mid-Scale",
        "run_label": "catboost_mid_scale",
        "status": "skipped",
        "fit_seconds": np.nan,
        "note": "RUN_CATBOOST_MID_SCALE is False.",
    })

persist_outputs()
model_status_df = pd.DataFrame(model_status_rows)
print("\nModel run status:")
print(model_status_df.to_string(index=False))

Mid-scale train sample rows: 300,000
Full test rows: 349,447
0:	test: 0.7022996	best: 0.7022996 (0)	total: 283ms	remaining: 56.2s
50:	test: 0.7166652	best: 0.7167339 (39)	total: 11.2s	remaining: 32.8s
100:	test: 0.7172802	best: 0.7172833 (99)	total: 24.6s	remaining: 24.1s
150:	test: 0.7175521	best: 0.7175655 (139)	total: 38.9s	remaining: 12.6s
199:	test: 0.7177039	best: 0.7177053 (198)	total: 52.5s	remaining: 0us

bestTest = 0.7177052956
bestIteration = 198

Shrink model to first 199 iterations.
CatBoost Mid-Scale numeric imputations (top 10):
            feature  fill_value  train_missing_before  eval_missing_before
                day     47.0000                     0                    0
              day_7      4.0000                     0                    0
             day_14      8.0000                     0                    0
             day_30     13.0000                     0                    0
             adFlag      0.0000                     0                    0


## Summary and Recommendation

The summary below reports only calculated values from successful challenger runs and compares them to the incumbent.

In [42]:
default_metrics_df = pd.read_csv(DEFAULT_METRICS_PATH)
threshold_results_df = pd.read_csv(THRESHOLD_RESULTS_PATH)
best_models_df = pd.read_csv(BEST_MODELS_PATH)
predictions_df = pd.read_csv(PREDICTIONS_PATH)
feature_importance_df = pd.read_csv(FEATURE_IMPORTANCE_PATH)

print("Model runtime overview:")
print(model_status_df.to_string(index=False))

if best_models_df.empty:
    print("\nNo CatBoost mid-scale model completed successfully.")
    print("Decision: b) HistGradientBoosting bleibt Gewinner")
else:
    best_models_df["interesting_vs_histgb"] = best_models_df["interesting_vs_histgb"].astype(str).str.lower().eq("true")
    best_models_df["precision_collapse"] = best_models_df["precision_collapse"].astype(str).str.lower().eq("true")

    top10_f1_df = rank_by_f1(threshold_results_df).head(10).reset_index(drop=True)
    top10_mcc_df = rank_by_mcc(threshold_results_df).head(10).reset_index(drop=True)
    best_challenger_row = rank_by_f1(best_models_df).iloc[0].copy()
    replacement_decision = catboost_midscale_replacement_decision(best_challenger_row, incumbent_row)

    delta_vs_incumbent = {
        "delta_auc": float(best_challenger_row["auc"]) - float(incumbent_row["auc"]),
        "delta_logloss": float(best_challenger_row["logloss"]) - float(incumbent_row["logloss"]),
        "delta_f1": float(best_challenger_row["f1"]) - float(incumbent_row["f1"]),
        "delta_precision": float(best_challenger_row["precision"]) - float(incumbent_row["precision"]),
        "delta_recall": float(best_challenger_row["recall"]) - float(incumbent_row["recall"]),
        "delta_mcc": float(best_challenger_row["mcc"]) - float(incumbent_row["mcc"]),
        "delta_tn": int(best_challenger_row["tn"]) - int(incumbent_row["tn"]),
        "delta_fp": int(best_challenger_row["fp"]) - int(incumbent_row["fp"]),
        "delta_fn": int(best_challenger_row["fn"]) - int(incumbent_row["fn"]),
        "delta_tp": int(best_challenger_row["tp"]) - int(incumbent_row["tp"]),
    }

    decision_text = (
        "a) CatBoost ersetzt HistGB"
        if replacement_decision["replace_histgb"]
        else "b) HistGradientBoosting bleibt Gewinner"
    )

    print("\nDefault threshold metrics (0.50):")
    print(default_metrics_df.sort_values(by=["f1", "mcc"], ascending=[False, False]).to_string(index=False))

    print("\nTop 10 by F1:")
    print(top10_f1_df.to_string(index=False))

    print("\nTop 10 by MCC:")
    print(top10_mcc_df.to_string(index=False))

    print("\nBest CatBoost mid-scale challenger:")
    print(pd.DataFrame([best_challenger_row]).to_string(index=False))

    print("\nBest challenger confusion matrix:")
    print(pd.DataFrame([{
        "TN": int(best_challenger_row["tn"]),
        "FP": int(best_challenger_row["fp"]),
        "FN": int(best_challenger_row["fn"]),
        "TP": int(best_challenger_row["tp"]),
    }]).to_string(index=False))

    print("\nDelta versus HistGradientBoosting incumbent:")
    print(pd.DataFrame([delta_vs_incumbent]).to_string(index=False))

    print("\nReplacement decision:")
    print(pd.DataFrame([replacement_decision]).to_string(index=False))
    print(f"Decision: {decision_text}")

    if not feature_importance_df.empty:
        print("\nTop 30 feature importances:")
        print(feature_importance_df.sort_values(by="importance", ascending=False).head(30).to_string(index=False))

    print("\nSaved files:")
    print(DEFAULT_METRICS_PATH)
    print(THRESHOLD_RESULTS_PATH)
    print(BEST_MODELS_PATH)
    print(PREDICTIONS_PATH)
    print(FEATURE_IMPORTANCE_PATH)

Model runtime overview:
             model          run_label  status  fit_seconds                                                                                                                                                          note
    LightGBM Quick     lightgbm_quick skipped          NaN                                                                                              LightGBM retraining is disabled for this CatBoost mid-scale run.
LightGBM Mid-Scale lightgbm_mid_scale skipped          NaN                                                                                               LightGBM mid-scale is disabled for this CatBoost mid-scale run.
    CatBoost Quick     catboost_quick skipped          NaN                                                                                                   CatBoost quick is disabled for this CatBoost mid-scale run.
CatBoost Mid-Scale catboost_mid_scale success    54.760215 CatBoost Mid-Scale does not clear the incumbent r

## Assertions

These checks verify the train/test-only contract, leakage removal, output persistence, and threshold-grid completeness.

In [43]:
assert TARGET_COLUMN in train_df.columns and TARGET_COLUMN in test_df.columns
assert set(train_df[TARGET_COLUMN].dropna().astype(int).unique()) == {0, 1}
assert set(test_df[TARGET_COLUMN].dropna().astype(int).unique()) == {0, 1}
assert feature_columns == [column for column in test_df.columns if column != TARGET_COLUMN and column not in LEAKAGE_COLUMNS]
assert all(column not in feature_columns for column in LEAKAGE_COLUMNS)

default_metrics_df = pd.read_csv(DEFAULT_METRICS_PATH)
threshold_results_df = pd.read_csv(THRESHOLD_RESULTS_PATH)
best_models_df = pd.read_csv(BEST_MODELS_PATH)

if not threshold_results_df.empty:
    threshold_counts = threshold_results_df.groupby("run_label")["threshold"].nunique()
    assert (threshold_counts == EXPECTED_THRESHOLD_COUNT).all(), threshold_counts.to_dict()

for output_path in [
    DEFAULT_METRICS_PATH,
    THRESHOLD_RESULTS_PATH,
    BEST_MODELS_PATH,
    PREDICTIONS_PATH,
    FEATURE_IMPORTANCE_PATH,
]:
    assert output_path.exists(), f"Missing output file: {output_path}"

assert "val_df" not in globals()
print("All quick-challenger assertions passed.")

All quick-challenger assertions passed.


## FINAL HOLDOUT VALIDATION — LOCKED

This notebook never loads `cls_val.csv`. The holdout remains explicitly blocked.

In [38]:
if not RUN_FINAL_VAL:
    print("Final VAL is locked. cls_val.csv is not loaded in this notebook.")
else:
    raise NotImplementedError("VAL remains intentionally locked in quick_challenger_lgbm_catboost.ipynb.")

Final VAL is locked. cls_val.csv is not loaded in this notebook.
